In [1]:
%load_ext autoreload
%autoreload 2

# M5 Forecast

## Imports

In [ ]:
import time
import os
import warnings
import numpy as np
import multiprocessing
import pandas as pd
from pathlib import Path
from loguru import logger
from m5.config import SETTINGS

# Silence cosmetic SyntaxWarnings from statsforecast / utilsforecast docstrings
# (raw-string oversight upstream — fires at compile time, no relation to our code).
warnings.filterwarnings("ignore", category=SyntaxWarning)

SETTINGS.ensure_dirs()

# Knobs sourced from .env (see README → Configuration). Override per-run via env vars,
# e.g. `M5_HORIZON=14 M5_N_WINDOWS=1 jupyter lab`.
HORIZON  = SETTINGS.horizon              # M5_HORIZON
VAL_DAYS = 0                             # held-out days for in-notebook validation (not in .env)

PATH_INPUT = SETTINGS.raw_dir            # DATA_DIR / m5 / datasets

id_col = "id"
time_col = "date"
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

N_JOBS = multiprocessing.cpu_count()

if not PATH_INPUT.exists():
    raise FileNotFoundError(
        f"M5 dataset not found at {PATH_INPUT}. "
        "Run `make download` (or `uv run m5 download`) to fetch it."
    )

os.environ["NIXTLA_ID_AS_COL"] = '1'
start_time = time.time()

# Helper Functions

In [ ]:
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import ExpandingMean, RollingMean, SeasonalRollingMean
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mse, mase, mape, mae, smape
from utilsforecast.plotting import plot_series
from utilsforecast.preprocessing import fill_gaps
from m5.legacy import *  # backwards-compat shim; new code should import from m5.data / m5.config


def is_first_or_fifteenth(dates):
    """Date is the 1st or 15th of the month (mlforecast date_features callable)."""
    return dates.day.isin([1, 15])


def evaluate_cross_validation(
    df: pd.DataFrame,
    metric: callable,
    id_col: str = "unique_id",
    time_col: str = "ds",
) -> pd.DataFrame:
    """Average a metric across CV cutoffs and pick best model per series."""
    models = df.drop(columns=[id_col, "ds", "cutoff", "y"]).columns.tolist()
    evals = []
    for cutoff in df["cutoff"].unique():
        eval_ = evaluate(
            df[df["cutoff"] == cutoff],
            metrics=[metric],
            models=models,
            id_col=id_col,
            time_col=time_col,
        )
        evals.append(eval_)
    evals = pd.concat(evals)
    evals = evals.groupby(id_col).mean(numeric_only=True)
    evals["best_model"] = evals.idxmin(axis=1)
    return evals


def get_best_model_forecast(
    forecasts_df: pd.DataFrame,
    evaluation_df: pd.DataFrame,
    id_col: str = "unique_id",
) -> pd.DataFrame:
    """Per-series production forecast using the best model identified by CV."""
    df = forecasts_df.set_index([id_col, "ds"]).stack().to_frame().reset_index(level=2)
    df.columns = ["model", "best_model_forecast"]
    df = df.join(evaluation_df[["best_model"]])
    df = df[df["best_model"] == df["model"]].reset_index().drop("model", axis=1)
    return df

# Load Inputs

In [ ]:
df = load_train_parquet()  # legacy shim → reads data/processed/long.parquet and renames unique_id→id, ds→date

# Model

In [ ]:
# Future Covariates
X_cols = ['sell_price', 'id', 'date', 'event_name_1', 'event_type_1',
          'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

In [ ]:
df.tail()

In [ ]:
df.date.max()

# Statistical Forecast
This is just a quick exploration, implement `Naive` forecast to set a benchmark for performance. There are too many time-series for the statistical models to use as a starting point.

- [statsforecast - nixtla](https://)
- [naive-methods](https://otexts.com/fpp3/simple-methods.html)
- [complex-exponential smoothing](https://onlinelibrary.wiley.com/doi/full/10.1002/nav.22074)

In [ ]:
from statsforecast.models import (
    Naive, 
    WindowAverage,
    SeasonalNaive,

    # Statistical Models
    AutoARIMA, AutoETS, AutoCES,
    
    # Itermittent Demand Models
    CrostonOptimized,
    CrostonSBA,
    CrostonClassic as Croston,
    # IMAPA,
)
from statsforecast.models import IMAPA
from statsforecast import StatsForecast


K = 50   # How to choose seasonality for daily data

model_list = [Naive(),
              WindowAverage(window_size=7 * 1, alias="1WeekMA"),
              WindowAverage(window_size=7 * 2, alias="2WeekMA"),
              WindowAverage(window_size=7 * 4, alias="4WeekMA"),

              # Itermittment Demand Optimized Models
              Croston(),
              CrostonOptimized(),
              CrostonSBA(),

              # # IMAPA(),
              # These take far too long
              # AutoCES(season_length=K),
              # AutoETS(season_length=K),
              AutoETS(season_length=7, model='ZNA'),
             ]

sf = StatsForecast(models=model_list, freq="D", n_jobs=N_JOBS)

# Cross-Validation

In [ ]:
%%time

CV_WINDOWS = SETTINGS.n_windows  # M5_N_WINDOWS

stat_cv = sf.cross_validation(h=HORIZON,
                              n_windows=CV_WINDOWS,
                              df=df,
                              id_col=id_col, time_col=time_col
                             )

stat_cv.tail()

In [ ]:
stat_preds

In [ ]:
stat_preds.to_parquet(SETTINGS.forecasts_dir / "stat_baseline.snap.parquet")

In [ ]:
stat_preds

In [ ]:
%%time

sf.fit(df, id_col=id_col, time_col=time_col)
stat_preds = sf.predict(HORIZON)
stat_preds

In [ ]:
stat_model_list = [str(m) for m in sf.models]
stat_model_list

In [ ]:
TRAIN_START = 1
TRAIN_END = 1941

TEST_START = 1942
TEST_END = 1969
TEST_DAYS = [f"d_{x}" for x in range(TEST_START, TEST_END + 1)]

from datasetsforecast.m5 import M5Evaluation
truth = pd.read_csv(PATH_INPUT / "sales_test_evaluation.csv")
truth["id"] = truth["item_id"].astype(str) + "_" + truth["store_id"].astype(str) + "_evaluation"


def make_submission(preds: pd.DataFrame, h: int, *, format: str = "kaggle") -> pd.DataFrame:
    """Pivot a long forecast frame into a wide submission table.

    format='kaggle'  → columns F1..Fh           (M5Evaluation / Kaggle layout)
    format='d_index' → columns d_{TEST_START+i} (raw M5 day-index layout)
    """
    wide = preds.pivot_table(index='id', columns='date', observed=True)
    if format == "kaggle":
        wide.columns = [f'F{i+1}' for i in range(h)]
    elif format == "d_index":
        wide.columns = [f'd_{TEST_START + i}' for i in range(h)]
    else:
        raise ValueError(f"Unknown format: {format!r}. Use 'kaggle' or 'd_index'.")
    wide.columns.name = None
    wide.index.name = 'id'
    return wide


In [ ]:
stat_models = sf.models

for model in stat_model_list:
    preds_long = stat_preds[["id", "date", str(model)]]
    df_sub = make_submission(preds_long, HORIZON)
    df_sub = truth[id_cols].merge(df_sub, on=["id"])
    model_score = M5Evaluation.evaluate(str(SETTINGS.data_dir), df_sub)
    model_score_avg = model_score["wrmsse"].mean()
    logger.info(f"""{model} scored: {model_score_avg:.3f}""")


In [ ]:
%%time

from tqdm import tqdm

stat_models = sf.models

stat_results = []
for model in tqdm(sf.models):
    model = str(model)
    preds_long = stat_preds[["id", "date", model]].copy()
    preds_long[model] = preds_long[model].round(0).clip(0)
    df_sub = make_submission(preds_long, HORIZON)
    df_sub = truth[id_cols].merge(df_sub, on=["id"])
    model_score = M5Evaluation.evaluate(str(SETTINGS.data_dir), df_sub)
    model_score["model"] = model
    model_score_avg = model_score["wrmsse"].mean()
    stat_results.append(model_score)
    logger.info(f"""{model} scored: {model_score_avg:.3f}""")

In [ ]:
dfresults_stats = pd.concat(stat_results)
dfresults_stats

In [ ]:
dfresults_stats.groupby("model")["wrmsse"].mean().sort_values()

# LightGbm CV

Fit the above MLForecast with lgb.LGBMRegressor parameters `lgb_params` specified.

- FREQ: `D` daily
- LAG_FEATURES: Calculates a list of `n` period with (freq) lag (i.e. 1, 7 would be y_1 and y_7 equal to yesterday and today. 
- LAG_TRANSFORM_DICT: Specify dictionary of tranformations to apply against `LAG_FEATURES`

- `lgb_params`: parameter dictionary for lightgbm
    - `num_leaves` and `n_estimators` are the key ones handling regularization and complexity

In [ ]:
lgb_params = {
    'verbose': 1,
    'num_threads': 4,
    'force_col_wise': True,
    'num_leaves': 256,
    'n_estimators': 100,
}

N_WEEKLY_LAGS = 8
WEEKLY_LAG_FEATURES = [7 * (i+1) for i in range(8)]
LAG_FEATURES = [1, 2, 3, ] + WEEKLY_LAG_FEATURES

LAG_TRANSFORM_DICT = {
    1:  [ExpandingMean()],
    7:  [RollingMean(7), SeasonalRollingMean(7, 1)],    # Capture weekly trend ajusting for seasonality (day of week effects)
    14: [RollingMean(14), SeasonalRollingMean(14, 1)],  # Capture bi-weekly trend adjusting for seasonality (i.e. payday)
    28: [RollingMean(28), SeasonalRollingMean(28, 1)],  # Capture Monthly trend adjusting for monthly seasonality
}

# List of built-in date features to compute with MLForecast.date_features
DATE_FEATURE_LIST = [
        'year', 'month', 'day', 'day_of_week', 'quarter', 'week',
        'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end',
        is_first_or_fifteenth,
    ]

In [ ]:
%%time

fcst = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq='D',
    lags=LAG_FEATURES,
    lag_transforms = LAG_TRANSFORM_DICT,
    date_features = DATE_FEATURE_LIST,
    num_threads=4,
)

PARAMS = fcst.models["LGBMRegressor"].get_params()

lgb_cv = fcst.cross_validation(h=HORIZON,
                               n_windows=1,
                               df=df,
                               static_features=id_cols,
                               id_col=id_col, time_col=time_col
                              )

## Combine CV Results

- `stat_cv` - DataFrame of stats models cross-validated predictions
- `lgb_cv` - DataFrame of lightgbm cross-validated predictions
- `df_cv` - DataFrame of combined cv results all forecasts

In [ ]:
df_cv = stat_cv.merge(lgb_cv[[id_col, time_col, "cutoff", "LGBMRegressor"]],
                      on=[id_col, time_col, "cutoff"])
df_cv = df_cv.rename(columns={time_col: "ds"})

MODEL_NAMES = df_cv.drop(["id", "ds", "cutoff", "y"], axis="columns").columns.tolist()

df_cv[MODEL_NAMES] = df_cv[MODEL_NAMES].round(0)  # round forecast to nearest integer

cv_err = evaluate_cross_validation(df_cv, id_col=id_col,
                                   # metric=smape
                                   metric=mae
                                  )
cv_err["LGBMRegressor"] = cv_err["LGBMRegressor"].round(0)
bst = get_best_model_forecast(df_cv, cv_err, id_col)
assert int(bst.groupby("id", observed=True).agg({"best_model": "nunique"}).max().iloc[0]) == 1
bst.best_model.value_counts(normalize=True)

# Fit LightGbm on Full Data

In [ ]:
%%time

lgb_start_time = time.time()

last_date_train = df['date'].max()
last_wmyrwk = df[df['date'] < last_date_train]['wm_yr_wk'].max()
logger.info(f"""Begin fitting LightGbm with {df[id_col].nunique():,d} ids data before: {last_date_train.date()}""")

logger.info(f"""LGB Fit Parameters:\n{PARAMS}""")
fcst.fit(
    df,
    id_col=id_col,
    time_col=time_col,
    target_col='y',
    static_features=id_cols,
)

logger.info(f"""Finished fitting LightGbm on {df["id"].nunique():,d} time-series in {time.time()- lgb_start_time:,.2f} seconds.""")

In [ ]:
## Create Future Regressors

cal = load_calendar(PATH_INPUT)
prices = load_prices(PATH_INPUT)

X_df = create_future_features(last_date_train, cal, prices, HORIZON)
X_df = X_df.merge(cal[["date", "d", "wm_yr_wk"]], on='date', how='left')

In [ ]:
%%time
preds = fcst.predict(HORIZON, X_df=X_df)
logger.info(f"""Finished generating {HORIZON:,d} days of predictions for {preds["id"].nunique():,d} time-series in {time.time()- lgb_start_time:,.2f} seconds.""")

In [ ]:
TEST_START 

# Create Submission

In [ ]:
submission = make_submission(preds, HORIZON, format="d_index")
submission = submission.join(df[id_cols].set_index("id").drop_duplicates())
submission.reset_index().rename(columns={"id": "unique_id"}).to_parquet(
    SETTINGS.forecasts_dir / "ml_baseline.snap.parquet"
)

In [ ]:
submission

In [ ]:
print(f"""Notebook finished in {(time.time() - start_time) / 60:.2f}m""")